In [ ]:
from pathlib import Path

# The project root, where credentials.json lives. Notebooks run from /notebooks, so we point up
ROOT = Path.home() / "Projects-COD" / "inbox-agent"
CREDENTIALS_FILE = ROOT / "credentials.json"   # The app's ID card (from Google Cloud)
TOKEN_FILE = ROOT / "token.json"               # Your saved login, created on first sign-in

# The permission we ask for: read-only Gmail. Least privilege
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

print("ID card found:", CREDENTIALS_FILE.exists())   # Should be True

In [ ]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

def get_credentials():
    """Return valid Google credentials, signing in through the browser only if needed."""
    creds = None

    # If you've signed in before, reuse the saved login
    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # The short-term pass expired: quietly get a new one, no browser needed
            creds.refresh(Request())
        else:
            # First time: open the browser so you can sign in and approve
            flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES)
            creds = flow.run_local_server(port=0)   # port=0 means pick any free port on your laptop
        # Save the login so next time there's no browser step
        TOKEN_FILE.write_text(creds.to_json())

    return creds

creds = get_credentials()
print("Signed in:", creds.valid)

In [ ]:
from googleapiclient.discovery import build

# Create a connection to Gmail using your login
service = build("gmail", "v1", credentials=creds)

# .execute() actually sends the request. Before that, it's just prepared, like writing a query before running it
profile = service.users().getProfile(userId="me").execute()
inbox = service.users().labels().get(userId="me", id="INBOX").execute()

print("Signed in as:", profile["emailAddress"])
print("Unread emails in inbox:", inbox["messagesUnread"])
print("Unread conversations in inbox:", inbox["threadsUnread"])

In [ ]:
'''
#CELL 4 AS MENTIONED IN CLAUDE CHAT
import html

def fetch_unread(service, max_results=10):
    """Return a list of unread inbox emails as simple dictionaries."""

    # Call 1 (LIST): ask Gmail for the IDs of unread inbox emails.
    # q uses the same syntax as Gmail's search bar
    response = service.users().messages().list(
        userId="me", q="is:unread in:inbox", maxResults=max_results
    ).execute()

    # If there are no unread emails, the "messages" key is missing, so default to an empty list
    messages = response.get("messages", [])

    records = []
    for msg in messages:
        # Call 2 (GET): open one email by its ID. Headers only, no body: faster, and we pull only what we need
        full = service.users().messages().get(
            userId="me",
            id=msg["id"],
            format="metadata",
            metadataHeaders=["From", "Subject", "Date"],
        ).execute()

        # Headers arrive as a list of {"name": ..., "value": ...} pairs.
        # Turn them into a lookup table so we can ask for headers["Subject"] directly
        headers = {h["name"]: h["value"] for h in full["payload"].get("headers", [])}

        # Build one clean record. .get(key, default) avoids crashes when something is missing
        records.append({
            "id": full["id"],                                        # Needed later to mark it as read
            "thread_id": full["threadId"],                           # Groups replies into one conversation
            "sender": headers.get("From", "(unknown sender)"),
            "subject": headers.get("Subject", "(no subject)"),
            "date": headers.get("Date", ""),
            "snippet": html.unescape(full.get("snippet", "")),      # Gmail encodes ' as &#39;; this decodes it
            "labels": full.get("labelIds", []),                     # e.g. UNREAD, INBOX, CATEGORY_PROMOTIONS
        })

    return records  '''

In [ ]:
# ---- SETUP: tools and settings used by the email-body functions below ----

import base64                  # Unpacks Gmail's "packing code" (base64) into readable text
import html                    # Fixes encoded symbols, e.g. &#39; back into '
import re                      # Pattern-based find-and-replace (like Excel SUBSTITUTE, but matches patterns)
from bs4 import BeautifulSoup  # Reads HTML and keeps only the visible text, dropping the tags

# Max characters of body we keep per email (~2,000 tokens) to control cost.
# Older quoted replies sit at the bottom of an email, so they're the first to get cut.
# CAPITALS = a setting by convention, not changed while the code runs.
MAX_BODY_CHARS = 8000

In [ ]:
def _decode(data):
    """Turn Gmail's encoded text (URL-safe base64) back into normal readable text."""
    # The leading _ in the name means: a helper for other functions, not called directly.

    # Base64 works in blocks of 4 characters, and Gmail sometimes drops the "=" fillers at the end.
    # (-len(data) % 4) works out how many are missing (0 to 3), and we add them back.
    padded = data + "=" * (-len(data) % 4)

    # Unpack the code into raw bytes (computer-level data, not yet readable).
    # "urlsafe" is the variant Gmail uses: it swaps + and / for - and _ so it's safe in web addresses.
    raw_bytes = base64.urlsafe_b64decode(padded)

    # Turn bytes into text using utf-8 (the standard text format).
    # errors="replace": a broken character becomes a placeholder instead of crashing the whole run.
    return raw_bytes.decode("utf-8", errors="replace")

In [ ]:
def _find_part(payload, mime_type):
    """Search an email's nested parts and return the encoded content of the FIRST part
    of the requested type (e.g. "text/plain"), skipping attachments. Returns None if not found.

    payload   = one "doll": a dictionary describing one part of the email
    mime_type = the kind of content wanted, e.g. "text/plain" or "text/html"
    """

    # ---- Step 1: check THIS doll ----
    is_right_type = payload.get("mimeType") == mime_type   # Is it the kind of content we want?
    is_attachment = bool(payload.get("filename"))          # Attachments have a filename (e.g. report.pdf); the email's own text doesn't
    data = payload.get("body", {}).get("data")             # The encoded content; .get(..., {}) avoids a crash if "body" is missing

    if is_right_type and not is_attachment and data:
        return data   # Found it. Return it still encoded; _decode unpacks it later

    # ---- Step 2: not this doll, so open each inner doll and search inside ----
    # The function calls itself on each inner part ("recursion"), so it handles any depth of nesting
    for part in payload.get("parts", []):     # .get("parts", []) = no inner dolls means nothing to loop over
        found = _find_part(part, mime_type)
        if found:
            return found   # Stop at the first match

    # ---- Step 3: searched everything, no match ----
    return None

In [ ]:
#TESTING OF THE ABOVE FUNCTIONS
# A fake email shaped like a real one: a plain-text version, an HTML version, and a PDF attachment.
# The "data" values are in Gmail's packing code, the same as real emails.
fake_email = {
    "mimeType": "multipart/mixed",
    "parts": [
        {"mimeType": "multipart/alternative", "parts": [
            {"mimeType": "text/plain", "body": {"data": "SGVsbG8gU2hhdW5haw"}},   # "Hello Shaunak"
            {"mimeType": "text/html",  "body": {"data": "PGI-SGk8L2I-"}},         # "<b>Hi</b>"
        ]},
        {"mimeType": "application/pdf", "filename": "report.pdf", "body": {"attachmentId": "abc"}},
    ],
}

print(_decode(_find_part(fake_email, "text/plain")))   # Expect: Hello Shaunak  (found two dolls deep)
print(_decode(_find_part(fake_email, "text/html")))    # Expect: <b>Hi</b>      (the HTML version)
print(_find_part(fake_email, "image/png"))             # Expect: None           (no such part exists)

In [ ]:
def extract_body(payload):
    """Return an email's readable text.
    Prefers the plain-text version; falls back to the HTML version with tags stripped.
    Returns "" if the email has no text at all (e.g. an image-only email).

    payload = the email's outer doll: full["payload"] from Gmail's get call
    Uses: _find_part (Piece 3) to locate the text, _decode (Piece 2) to unpack it.
    """

    # ---- Option 1: plain text (cleanest; most emails from real people include it) ----
    plain = _find_part(payload, "text/plain")
    if plain:
        text = _decode(plain)

    else:
        # ---- Option 2: HTML only (common for newsletters and marketing emails) ----
        html_data = _find_part(payload, "text/html")
        if not html_data:
            return ""   # No text in any form, so return an empty string rather than crashing

        # Parse the HTML and keep only the visible text, dropping the tags.
        # separator="\n" puts a line break between pieces of text so paragraphs don't run together.
        # Known side effect: inline tags like <b> can split one sentence across lines. Harmless for the LLM.
        text = BeautifulSoup(_decode(html_data), "html.parser").get_text(separator="\n")

    # ---- Clean-up: collapse runs of blank lines into a single blank line ----
    # Pattern \n\s*\n+ means: a line break, then any spaces or line breaks, then more line breaks.
    # Every match gets replaced with exactly one blank line ("\n\n").
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()   # Remove stray spaces and line breaks at the very start and end

In [ ]:
#TESTING EXTRACT BODY FUNCTION ABOVE
def _encode(text):
    """Test helper: the reverse of _decode, so we can build fake emails from normal text."""
    return base64.urlsafe_b64encode(text.encode("utf-8")).decode("ascii")

# Fake email 2: HTML only, with a big gap of blank lines inside
html_only_email = {
    "mimeType": "text/html",
    "body": {"data": _encode("<p>Hi Shaunak,</p>\n\n\n\n<p>See you <b>Thursday</b>.</p>")},
}

# Fake email 3: no text at all, just an image attachment
no_text_email = {
    "mimeType": "multipart/mixed",
    "parts": [{"mimeType": "image/png", "filename": "photo.png", "body": {"attachmentId": "xyz"}}],
}

# repr() shows the text exactly as stored, with invisible line breaks displayed as \n
print(repr(extract_body(fake_email)))        # fake_email from the Piece 3 test: plain text should win
print(repr(extract_body(html_only_email)))   # HTML fallback: tags gone, gap collapsed
print(repr(extract_body(no_text_email)))     # No text anywhere

In [ ]:
def fetch_unread(service, max_results=10):
    """Fetch unread inbox emails and return them as a list of clean dictionaries (one per email).

    service     = the Gmail connection created in Cell 3
    max_results = the most emails to fetch (keep it small while testing)
    Uses: extract_body (Piece 4) to get each email's readable text.
    """

    # ---- Call 1 (LIST): get the IDs of unread inbox emails. IDs only, no content ----
    response = service.users().messages().list(
        userId="me",              # "me" = the signed-in account
        q="is:unread in:inbox",   # Same search syntax as Gmail's search bar
        maxResults=max_results,
    ).execute()                   # .execute() actually sends the request

    # If nothing is unread, Gmail leaves out the "messages" key entirely, so default to an empty list
    messages = response.get("messages", [])

    records = []
    for msg in messages:
        # ---- Call 2 (GET): open one email by its ID ----
        # format="full" returns every header plus the body parts (the Russian dolls)
        full = service.users().messages().get(userId="me", id=msg["id"], format="full").execute()

        # Headers arrive as a list of {"name": ..., "value": ...} pairs. Turn them into a lookup table.
        # .lower() makes every name lowercase, because senders write "Cc", "CC", or "cc" inconsistently
        headers = {h["name"].lower(): h["value"] for h in full["payload"].get("headers", [])}

        body = extract_body(full["payload"])   # Readable text, before the cap

        # ---- Build one clean record. .get(key, default) fills a default instead of crashing ----
        records.append({
            "id": full["id"],                                    # Needed later to mark it read or draft a reply
            "thread_id": full["threadId"],                       # Links an email to the rest of its conversation
            "sender": headers.get("from", "(unknown sender)"),
            "to": headers.get("to", ""),                         # Was I addressed directly...
            "cc": headers.get("cc", ""),                         # ...or just copied?
            "subject": headers.get("subject", "(no subject)"),
            "date": headers.get("date", ""),
            "is_mass_mail": "list-unsubscribe" in headers,       # Newsletters and marketing almost always include this
            "snippet": html.unescape(full.get("snippet", "")),   # Gmail's preview line, with &#39; etc. decoded
            "body": body[:MAX_BODY_CHARS],                       # [:8000] = keep the first 8,000 characters (cuts oldest quoted replies)
            "body_truncated": len(body) > MAX_BODY_CHARS,        # Record that we cut it, so we never claim we saw everything
            "labels": full.get("labelIds", []),                  # e.g. UNREAD, INBOX, CATEGORY_PROMOTIONS
        })

    return records

In [ ]:
import time

# ---- Fetch and time it (latency matters as volume grows) ----
start = time.perf_counter()
records = fetch_unread(service, max_results=10)
elapsed = time.perf_counter() - start
print(f"Fetched {len(records)} emails in {elapsed:.1f}s\n")

# ---- One line per email: mass-mail flag | sender | subject | body length ----
for i, r in enumerate(records):                        # enumerate adds a counter i (0, 1, 2...) to each email
    flag = "MASS" if r["is_mass_mail"] else "    "
    cut = " (truncated)" if r["body_truncated"] else ""
    print(f"{i:2} | {flag} | {r['sender'][:28]:28} | {r['subject'][:40]:40} | {len(r['body'])} chars{cut}")

# ---- Read one body in full to check the cleaning. Change SAMPLE to look at a different email ----
SAMPLE = 0
print(f"\n--- Body of email {SAMPLE} (first 1,500 characters) ---\n")
print(records[SAMPLE]["body"][:1500] if records else "No unread emails")

In [ ]:
#USELESS, JUST TO SHOW LIST CONTAINS ONLY IDS NOT AT ALL TEXT STUFF, YOU NEED TO GET TEXT SEPARATELY WHICH IS IN THE FUNCITON
print(service.users().messages().list(userId="me", q="is:unread in:inbox", maxResults=20).execute())